# Two-converter EMT shared-network result

This notebook visualizes the 50 µs two-converter example. Both FMU instances are connected to the same native DPsim three-phase network. The CSV is preserved by the validation run so the plots can be regenerated without rerunning the simulation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_candidates = [
    Path(
        "../../build-macos-check/fmi-validation/two_converters_50us/logs/fmi_two_converters.csv"
    ),
    Path(
        "../../../build-macos-check/fmi-validation/two_converters_50us/logs/fmi_two_converters.csv"
    ),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find the preserved two-converter CSV")
data = pd.read_csv(csv_path, skipinitialspace=True)
data.columns = [column.strip() for column in data.columns]
data.head()

In [ ]:
time_ms = 1000.0 * data["time"]
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
phase_names = ["a", "b", "c"]
for phase, name in enumerate(phase_names):
    axes[0].plot(time_ms, data["pcc_voltage_" + str(phase)], label="phase " + name)
axes[0].set_ylabel("PCC voltage")
axes[0].set_title("Shared PCC voltage")
axes[0].legend()
for phase, name in enumerate(phase_names):
    axes[1].plot(
        time_ms, data["first_current_" + str(phase)], label="converter 1 phase " + name
    )
    axes[1].plot(
        time_ms,
        data["second_current_" + str(phase)],
        "--",
        label="converter 2 phase " + name,
    )
axes[1].set_xlabel("time [ms]")
axes[1].set_ylabel("current")
axes[1].set_title("Converter currents")
axes[1].legend(ncol=2)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].semilogy(
    time_ms, np.maximum(data["first_residual"], 1e-16), label="converter 1"
)
axes[0].semilogy(
    time_ms, np.maximum(data["second_residual"], 1e-16), "--", label="converter 2"
)
axes[0].set_ylabel("residual")
axes[0].set_title("Coupling residuals")
axes[0].legend()
axes[1].step(time_ms, data["first_internal_steps"], where="post", label="converter 1")
axes[1].step(
    time_ms,
    data["second_internal_steps"],
    where="post",
    linestyle="--",
    label="converter 2",
)
axes[1].set_xlabel("time [ms]")
axes[1].set_ylabel("internal steps")
axes[1].set_title("FMU internal-step activity")
axes[1].legend()
fig.tight_layout()

In [ ]:
first_current = data[
    ["first_current_0", "first_current_1", "first_current_2"]
].to_numpy()
second_current = data[
    ["second_current_0", "second_current_1", "second_current_2"]
].to_numpy()
current_difference = np.linalg.norm(first_current - second_current, axis=1)
print(f"rows: {len(data)}")
print(f'duration: {data["time"].iloc[-1]:.6g} s')
print(f"max current difference norm: {current_difference.max():.6g}")
print(f'max converter 1 residual: {data["first_residual"].max():.6g}')
print(f'max converter 2 residual: {data["second_residual"].max():.6g}')
print("converter 1 internal steps:", sorted(data["first_internal_steps"].unique()))
print("converter 2 internal steps:", sorted(data["second_internal_steps"].unique()))
plt.figure(figsize=(11, 3.5))
plt.plot(time_ms, current_difference)
plt.xlabel("time [ms]")
plt.ylabel("||i1 - i2||")
plt.title("Difference between the two identical converter responses")
plt.tight_layout()